In [ ]:
%pip install -r requirements.txt

In [7]:
import pandas as pd

In [8]:
annotations = pd.read_excel('dataset/anot.xlsx')
annotations = annotations.sort_values(by=['FILE ID','Start'], ascending=[True,True])

annotations = annotations.dropna(axis=0)

In [ ]:
%pip install mne

In [9]:
bipolar_pairs = [
    ('EEG Fp1-REF', 'EEG F7-REF'),
    ('EEG F7-REF',  'EEG T3-REF'),
    ('EEG T3-REF',  'EEG T5-REF'),
    ('EEG T5-REF',  'EEG O1-REF'),
    ('EEG Fp1-REF', 'EEG F3-REF'),
    ('EEG F3-REF',  'EEG C3-REF'),
    ('EEG C3-REF',  'EEG P3-REF'),
    ('EEG P3-REF',  'EEG O1-REF'),
    ('EEG Fz-REF',  'EEG Cz-REF'),
    ('EEG Cz-REF',  'EEG Pz-REF'),
    ('EEG Fp2-REF', 'EEG F4-REF'),
    ('EEG F4-REF',  'EEG C4-REF'),
    ('EEG C4-REF',  'EEG P4-REF'),
    ('EEG P4-REF',  'EEG O2-REF'),
    ('EEG Fp2-REF', 'EEG F8-REF'),
    ('EEG F8-REF',  'EEG T4-REF'),
    ('EEG T4-REF',  'EEG T6-REF'),
    ('EEG T6-REF',  'EEG O2-REF'),
]

desired_order = [
    'Fp2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'Fp1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'Fp2-F8', 'F8-T4', 'T4-T6', 'T6-O2',
    'Fp1-F7', 'F7-T3', 'T3-T5', 'T5-O1',
    'Fz-Cz', 'Cz-Pz',
]

def normalize_channel(ch):
    return ch.strip().upper()

bipolar_pairs = [(normalize_channel(a), normalize_channel(b)) for a, b in bipolar_pairs]


def pair_name(pair):
    left = pair[0].replace('EEG ', '').replace('-REF', '')
    right = pair[1].replace('EEG ', '').replace('-REF', '')
    return f"{left}-{right}".upper()  # Ensure uppercase for consistency

name_to_pair = {pair_name(p): p for p in bipolar_pairs}

# print("Available pairs in name_to_pair:")
# for name in name_to_pair.keys():
    # print(name)

desired_order_upper = [name.upper() for name in desired_order]

for name in desired_order_upper:
    if name not in name_to_pair:
        print(f"Warning: '{name}' not found in name_to_pair")

reordered_pairs = [name_to_pair[name] for name in desired_order_upper if name in name_to_pair]

def make_ch_names(pairs):
    def pretty(ch):
        ch = ch.replace('EEG ', '').replace('-REF', '')
        return ch.capitalize()
    return [f"{pretty(a)}-{pretty(b)}" for a, b in pairs]

ch_names = make_ch_names(reordered_pairs)

# for name, pair, pretty_name in zip(desired_order_upper, reordered_pairs, ch_names):
    # print(f"{name}: {pair} -> {pretty_name}")
anode = [a for a, _ in bipolar_pairs]
cathode = [b for _, b in bipolar_pairs]
ch_names = make_ch_names(reordered_pairs)

In [10]:
import mne

def getArray(filename: str):
    raw = mne.io.read_raw_edf(filename, preload=True)
    raw.rename_channels(lambda ch: ch.upper())
    
    drop_candidates = ['ECG EKG', 'RESP EFFORT', 'ECG EKG-REF', 'RESP EFFORT-REF']
    available = set(raw.ch_names)
    to_drop = [ch for ch in drop_candidates if ch in available]
    
    if to_drop:
        raw.drop_channels(to_drop)
    
    raw = mne.set_bipolar_reference(raw, anode=anode, cathode=cathode, ch_name=ch_names, copy=True)
    
    # events = mne.make_fixed_length_events(raw, id=1, duration=1.0, overlap=0)
    epochs = mne.make_fixed_length_epochs(raw, duration = 1)
    epoched_array = epochs.get_data()  # Shape: (n_epochs, n_channels, n_times)
    
    return epoched_array

In [11]:
import os

eeg_set = []
for i in range(79):
    filepath = f'dataset/eeg{i + 1}.edf'
    if os.path.exists(filepath):
        try:
            eeg = getArray(filepath)
            eeg_set.append(eeg)
        except ValueError as e:
            print(f"Error in file: {filepath}")
            print(e)

Extracting EDF parameters from /Users/adityakinjawadekar/Documents/svm/home/dataset/eeg1.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1790207  =      0.000 ...  6992.996 secs...
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=18, n_times=1790208
    Range : 0 ... 1790207 =      0.000 ...  6992.996 secs
Ready.
Added the following bipolar channels:
Fp2-F4, F4-C4, C4-P4, P4-O2, Fp1-F3, F3-C3, C3-P3, P3-O1, Fp2-F8, F8-T4, T4-T6, T6-O2, Fp1-F7, F7-T3, T3-T5, T5-O1, Fz-Cz, Cz-Pz
Not setting metadata
6993 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 6993 events and 256 original time points ...
0 bad epochs dropped
Extracting EDF parameters from /Users/adityakinjawadekar/Documents/svm/home/dataset/eeg2.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 962815  =      0.

In [12]:
annotations 

,FILE ID,Start,Stop,Fp2-F4,F4-C4,C4-P4,P4-O2,Fp1-F3,F3-C3,C3-P3,...,Fp2-F8,F8-T4,T4-T6,T6-O2,Fp1-F7,F7-T3,T3-T5,T5-O1,Fz-Cz,Cz-Pz
1,1.0,104.0,122.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
2,1.0,349.0,428.0,0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
3,1.0,760.0,779.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
4,1.0,956.0,965.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
5,1.0,973.0,982.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,79.0,152.0,190.0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
361,79.0,367.0,384.0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
362,79.0,576.0,621.0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
363,79.0,2148.0,2163.0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
eeg_set[0].shape

(6993, 18, 256)

In [13]:
channel_cols = ['Fp2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'Fp1-F3', 'F3-C3', 'C3-P3',
                'Fp2-F8', 'F8-T4', 'T4-T6', 'T6-O2', 'Fp1-F7', 'F7-T3', 'T3-T5', 
                'T5-O1', 'Fz-Cz', 'Cz-Pz']


In [ ]:
import csv
import numpy as np
from pathlib import Path

# Output directory for per-patient CSVs
output_dir = Path("annotated_epochs_csvs")
output_dir.mkdir(exist_ok=True)

# Sanitize FILE ID column to ensure it's integer and 0-based
file_ids = annotations['FILE ID'].astype(int) - 1  # now 0-indexed

for patient_id in range(len(eeg_set)):  # 0 to 78 (79 total)
    if patient_id == 50:
        continue

    # Filter annotations for current patient
    patient_annotations = annotations[file_ids == patient_id]

    if patient_annotations.empty:
        print(f"⚠️ No annotations found for patient {patient_id+1:03} — skipping file")
        continue

    csv_path = output_dir / f"annotated_epochs_patient_{patient_id+1:03}.csv"

    with open(csv_path, "w", newline='') as csvfile:
        writer = csv.writer(csvfile)
        header = ['label', 'channel'] + [f'sample_{i}' for i in range(256)]
        writer.writerow(header)

        for i, row in patient_annotations.iterrows():
            start_epoch = int(row['Start'])
            stop_epoch = int(row['Stop'])

            print(f"Patient {patient_id+1:03}: Annotation {i+1} — Epochs {start_epoch}–{stop_epoch}")

            for ch_idx, col in enumerate(channel_cols):
                if ch_idx >= eeg_set[patient_id].shape[1]:
                    continue  # skip missing channels

                label = int(row[col])
                channel_name = col

                for epoch_idx in range(start_epoch, stop_epoch):
                    epoch = eeg_set[patient_id][epoch_idx, ch_idx]
                    row_out = [label, channel_name] + epoch.tolist()
                    writer.writerow(row_out)


In [95]:
%pip install PyWavelets hurst

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 1.6 MB/s eta 0:00:0000:0100:010m

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import csv
import numpy as np
import pywt
from scipy.stats import entropy
from scipy.signal import welch
from pathlib import Path
import hurst
import warnings

warnings.filterwarnings("ignore")

# -------- Feature Extractors --------

def get_all_features_from_signal(sig):
    feats = {}
    feats['max'] = np.max(sig)
    feats['min'] = np.min(sig)
    feats['mean'] = np.mean(sig)
    feats['std'] = np.std(sig)
    
    try:
        feats['hurst'] = hurst.compute_Hc(sig, simplified=True)[0]
    except:
        feats['hurst'] = np.nan

    try:
        _, psd = welch(sig)
        psd_norm = psd / np.sum(psd)
        feats['entropy'] = entropy(psd_norm)
    except:
        feats['entropy'] = np.nan

    try:
        N = len(sig)
        L = np.sum(np.abs(np.diff(sig)))
        feats['fractal_dim'] = 1 + (np.log(N) / np.log(N / L)) if L > 0 else 1.0
    except:
        feats['fractal_dim'] = np.nan

    try:
        f, pxx = welch(sig, fs=256)
        feats['psd'] = np.mean(pxx)
    except:
        feats['psd'] = np.nan

    return feats

def get_all_dwt_features(epoch, wavelet='db4', level=4):
    coeffs = pywt.wavedec(epoch, wavelet=wavelet, level=level)
    names = ['a4', 'd4', 'd3', 'd2', 'd1']
    full_feat_dict = {}

    for name, sig in zip(names, coeffs):
        feat_dict = get_all_features_from_signal(sig)
        for stat_name, value in feat_dict.items():
            full_feat_dict[f"{name}_{stat_name}"] = value

    return full_feat_dict

# -------- Main Processing --------

output_dir = Path("annotated_feature_csvs")
output_dir.mkdir(exist_ok=True)

file_ids = annotations['FILE ID'].astype(int) - 1  # Ensure 0-indexed

for patient_id in range(len(eeg_set)):  # 0 to 78
    if patient_id == 50:
        continue  # Skip corrupted file if any

    patient_annotations = annotations[file_ids == patient_id]

    if patient_annotations.empty:
        print(f"⚠️ No annotations for patient {patient_id+1:03} — skipping file")
        continue

    csv_path = output_dir / f"features_patient_{patient_id+1:03}.csv"
    with open(csv_path, "w", newline='') as csvfile:
        writer = csv.writer(csvfile)

        # Dynamic feature column headers
        dwt_levels = ['a4', 'd4', 'd3', 'd2', 'd1']
        stats = ['max', 'min', 'mean', 'std', 'hurst', 'entropy', 'fractal_dim', 'psd']
        header = ['label', 'channel'] + [f"{lvl}_{stat}" for lvl in dwt_levels for stat in stats]
        writer.writerow(header)

        for i, row in patient_annotations.iterrows():
            start_epoch = int(row['Start'])
            stop_epoch = int(row['Stop'])

            print(f"Patient {patient_id+1:03}: Annotation {i+1} — Epochs {start_epoch}–{stop_epoch}")

            for ch_idx, col in enumerate(channel_cols):
                if ch_idx >= eeg_set[patient_id].shape[1]:
                    continue

                label = int(row[col])
                channel_name = col

                for epoch_idx in range(start_epoch, stop_epoch):
                    epoch = eeg_set[patient_id][epoch_idx, ch_idx]

                    features = get_all_dwt_features(epoch)
                    row_out = [label, channel_name] + [features[key] for key in header[2:]]
                    writer.writerow(row_out)


In [63]:
df = pd.read_csv('/Users/adityakinjawadekar/Documents/svm/home/annotated_feature_csvs/features_patient_001.csv')

In [64]:
df

,label,channel,sample_0,sample_1,sample_2,sample_3,sample_4,sample_5,sample_6,sample_7,...,sample_246,sample_247,sample_248,sample_249,sample_250,sample_251,sample_252,sample_253,sample_254,sample_255


In [14]:
import csv
import numpy as np
import pywt
from scipy.stats import entropy
from scipy.signal import welch
from pathlib import Path
import hurst
import warnings

warnings.filterwarnings("ignore")

# -------- Feature Functions --------

def get_all_features_from_signal(sig):
    feats = {}
    feats['max'] = np.max(sig)
    feats['min'] = np.min(sig)
    feats['mean'] = np.mean(sig)
    feats['std'] = np.std(sig)

    try:
        feats['hurst'] = hurst.compute_Hc(sig, simplified=True)[0]
    except:
        feats['hurst'] = np.nan

    try:
        _, psd = welch(sig)
        psd_norm = psd / np.sum(psd)
        feats['entropy'] = entropy(psd_norm)
    except:
        feats['entropy'] = np.nan

    try:
        N = len(sig)
        L = np.sum(np.abs(np.diff(sig)))
        feats['fractal_dim'] = 1 + (np.log(N) / np.log(N / L)) if L > 0 else 1.0
    except:
        feats['fractal_dim'] = np.nan

    try:
        f, pxx = welch(sig, fs=256)
        feats['psd'] = np.mean(pxx)
    except:
        feats['psd'] = np.nan

    return feats

def get_all_dwt_features(epoch, wavelet='db4', level=4):
    try:
        coeffs = pywt.wavedec(epoch, wavelet=wavelet, level=level)
    except:
        return None  # Skip this wavelet if it fails
    names = ['a4', 'd4', 'd3', 'd2', 'd1']
    full_feat_dict = {}
    for name, sig in zip(names, coeffs):
        feat_dict = get_all_features_from_signal(sig)
        for stat_name, value in feat_dict.items():
            full_feat_dict[f"{name}_{stat_name}"] = value
    return full_feat_dict

# -------- Main Loop --------

output_dir = Path("annotated_feature_csvs_all_wavelets")
output_dir.mkdir(exist_ok=True)

wavelets =  ['db4', 'sym5', 'bior3.5', 'coif3', 'morlet','haar']
file_ids = annotations['FILE ID'].astype(int) - 1

for wavelet in wavelets:
    print(f"\n🔷 Processing wavelet: {wavelet}")

    for patient_id in range(len(eeg_set)):  # 0 to 78
        if patient_id == 50:
            continue

        patient_annotations = annotations[file_ids == patient_id]
        if patient_annotations.empty:
            continue

        csv_path = output_dir / f"features_patient_{patient_id+1:03}_wavelet_{wavelet}.csv"
        with open(csv_path, "w", newline='') as csvfile:
            writer = csv.writer(csvfile)

            # Header
            dwt_levels = ['a4', 'd4', 'd3', 'd2', 'd1']
            stats = ['max', 'min', 'mean', 'std', 'hurst', 'entropy', 'fractal_dim', 'psd']
            header = ['label', 'channel'] + [f"{lvl}_{stat}" for lvl in dwt_levels for stat in stats]
            writer.writerow(header)

            for i, row in patient_annotations.iterrows():
                start_epoch = int(row['Start'])
                stop_epoch = int(row['Stop'])

                for ch_idx, col in enumerate(channel_cols):
                    if ch_idx >= eeg_set[patient_id].shape[1]:
                        continue

                    label = int(row[col])
                    channel_name = col

                    for epoch_idx in range(start_epoch, stop_epoch):
                        epoch = eeg_set[patient_id][epoch_idx, ch_idx]
                        features = get_all_dwt_features(epoch, wavelet=wavelet)

                        if features is None:
                            continue  # skip wavelets that fail for this epoch

                        row_out = [label, channel_name] + [features[key] for key in header[2:]]
                        writer.writerow(row_out)



🔷 Processing wavelet: db4

🔷 Processing wavelet: sym5

🔷 Processing wavelet: bior3.5

🔷 Processing wavelet: coif3

🔷 Processing wavelet: morlet

🔷 Processing wavelet: haar
